# 🥦 SoGood — Analyse Exploratoire des Données (EDA)
Dataset : Open Food Facts | Objectif : comprendre la qualité nutritionnelle des produits

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from config import CLEANED_CSV_PATH, FEATURE_COLUMNS

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

df = pd.read_csv(CLEANED_CSV_PATH)
print(f'Dataset : {len(df):,} produits | {df.shape[1]} colonnes')
df.head()

## 1. Vue d'ensemble

In [ ]:
print('=== Infos générales ===')
print(f"Produits      : {len(df):,}")
print(f"Marques       : {df['brands'].nunique():,}")
print(f"Catégories    : {df['categories_en'].nunique():,}")
print(f"Valeurs nulles:\n{df.isnull().sum()[df.isnull().sum()>0]}")

In [ ]:
# Distribution Nutri-Score
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'a':'#1fa37b','b':'#85bb2f','c':'#ffcc00','d':'#ff6600','e':'#ee3333'}
ns = df['nutriscore_grade'].value_counts().sort_index()
axes[0].bar(ns.index, ns.values, color=[colors[g] for g in ns.index], edgecolor='white')
axes[0].set_title('Distribution des Nutri-Scores', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Nutri-Score')
axes[0].set_ylabel('Nombre de produits')
for i, (k, v) in enumerate(ns.items()):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(ns.values, labels=[g.upper() for g in ns.index],
            colors=[colors[g] for g in ns.index],
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Répartition Nutri-Score', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/plot_nutriscore_dist.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Analyse des valeurs nutritionnelles

In [ ]:
num_cols = [c for c in FEATURE_COLUMNS if c in df.columns]
df[num_cols].describe().round(2)

In [ ]:
# Boxplots nutritionnels par Nutri-Score
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

grade_order = ['a','b','c','d','e']
palette = [colors[g] for g in grade_order]

for i, col in enumerate(num_cols[:9]):
    sns.boxplot(data=df, x='nutriscore_grade', y=col, order=grade_order,
                palette=palette, ax=axes[i], showfliers=False)
    axes[i].set_title(col.replace('_100g','').replace('-',' ').title())
    axes[i].set_xlabel('')

plt.suptitle('Valeurs nutritionnelles par Nutri-Score', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../data/plot_nutrition_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Matrice de corrélation
corr = df[num_cols].corr()
plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            mask=mask, square=True, linewidths=0.5)
plt.title('Corrélations entre valeurs nutritionnelles', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/plot_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Marques et produits controversés

In [ ]:
# Top marques avec le plus de produits D/E
bad = df[df['nutriscore_grade'].isin(['d','e'])]
top_bad = bad['brands'].value_counts().head(15)

plt.figure(figsize=(10, 6))
bars = plt.barh(top_bad.index[::-1], top_bad.values[::-1], color='#ee3333', alpha=0.8)
plt.xlabel('Nombre de produits Nutri-Score D/E')
plt.title('Top 15 marques avec le plus de produits controversés (D/E)', fontweight='bold')
for bar, val in zip(bars, top_bad.values[::-1]):
    plt.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
             str(val), va='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/plot_brands_controversial.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Produits les plus riches en sucre
print('=== Top 10 produits les plus sucrés ===')
df[['product_name','brands','nutriscore_grade','sugars_100g']]\
    .sort_values('sugars_100g', ascending=False).head(10)

In [ ]:
# Produits les plus salés
print('=== Top 10 produits les plus salés ===')
df[['product_name','brands','nutriscore_grade','salt_100g']]\
    .sort_values('salt_100g', ascending=False).head(10)

In [ ]:
# Produits avec le plus d'additifs
print('=== Top 10 produits avec le plus d\'additifs ===')
df[['product_name','brands','nutriscore_grade','additives_n']]\
    .sort_values('additives_n', ascending=False).head(10)

## 4. Analyse du modèle ML

In [ ]:
import joblib
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from config import MODEL_PATH, MODEL_DIR
import json

pipeline = joblib.load(MODEL_PATH)

cols = [c for c in FEATURE_COLUMNS if c in df.columns]
X = df[cols].values
y = df['nutriscore_label'].values
_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

y_pred = pipeline.predict(X_test)

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(cm, display_labels=['A','B','C','D','E'])
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Matrice de confusion — XGBoost', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/plot_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(classification_report(y_test, y_pred, target_names=['A','B','C','D','E']))

In [ ]:
# Feature importance
xgb_model = pipeline.named_steps['model']
fi = xgb_model.feature_importances_

plt.figure(figsize=(10, 5))
sorted_idx = fi.argsort()
plt.barh([cols[i] for i in sorted_idx], fi[sorted_idx], color='#1D7A5E', alpha=0.85)
plt.xlabel('Importance')
plt.title('Feature Importance — XGBoost', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/plot_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Comparaison des métriques modèles
metrics = {}
m1_path = MODEL_DIR / 'metrics.json'
m2_path = MODEL_DIR / 'nlp_metrics.json'

if m1_path.exists():
    metrics['XGBoost (valeurs nutritionnelles)'] = json.load(open(m1_path))
if m2_path.exists():
    metrics['NLP Hugging Face (texte ingrédients)'] = json.load(open(m2_path))

if metrics:
    fig, ax = plt.subplots(figsize=(9, 4))
    models = list(metrics.keys())
    accs   = [metrics[m]['accuracy']*100 for m in models]
    f1s    = [metrics[m]['f1_score'] for m in models]
    x = np.arange(len(models))
    bars1 = ax.bar(x - 0.2, accs, 0.35, label='Accuracy (%)', color='#1D7A5E', alpha=0.85)
    bars2 = ax.bar(x + 0.2, [f*100 for f in f1s], 0.35, label='F1-score (%)', color='#ff6600', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(models, wrap=True)
    ax.set_ylim(0, 100)
    ax.set_ylabel('Score (%)')
    ax.set_title('Comparaison des modèles', fontsize=14, fontweight='bold')
    ax.legend()
    for bar in bars1:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                f'{bar.get_height():.1f}%', ha='center', fontsize=9, fontweight='bold')
    for bar in bars2:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                f'{bar.get_height():.1f}%', ha='center', fontsize=9, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../data/plot_model_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

print('\n=== Résumé des performances ===')
for name, m in metrics.items():
    print(f"{name}: Accuracy={m['accuracy']*100:.1f}% | F1={m['f1_score']:.4f}")